# Using Pydantic to read .env parameters

Intent - Store DB Password, API Secrets into '.env' file. Pydantic-settings library lets you read these files automatically and converts strings into proper data types (like text to integer)
We will use two specific classes named 'BaseSettings' and 'SettingsConfigDict' into this demo



[1] - Installation - we must install pydantic-settings. Execute below command

#uv pip install pydantic-settings[=='version' (if required)]

or

#pip install pydantic-settings[=='version' (if required)]


In [2]:
# Import required libraries and defining the base class
import os
from pathlib import Path
from pydantic import Field, SecretStr, HttpUrl
from pydantic_settings import BaseSettings, SettingsConfigDict
from typing import Literal

# Dynamically find the path for '.env' file
# for normal python file having '.py' extention use below line 
# [uncomment it and comment jupyter notebook specific line]

# base_dir = Path(__file__).resolve().parent

# Jupyter notebook fix [comment it out when working with normal python file]
base_dir = Path(os.getcwd())

class AppConfig(BaseSettings):
    # Reading the '.env' file and load the values using Field(alias)
    automation_env: Literal["sbx", "dev", "tst", "prd"] = Field(alias="AUTOMATION_ENV")
    central_port: int = Field(alias="CENTRAL_PORT", ge=1024, le=65535)
    cloud_timeout: float = Field(alias="CLOUD_TIMEOUT", default=5.0)

    # Loading the URL
    slack_webhook_url: HttpUrl = Field(alias="SLACK_WEBHOOK_URL")

    # Loading Secret safely
    cloud_access_token: SecretStr = Field(alias="CLOUD_ACCESS_TOKEN")

    # Telling pydantic how and where to load the configuration
    model_config = SettingsConfigDict(
        env_file= base_dir / '.env',               # Will look for this specific file name
        env_file_encoding='utf-8',     # Read text using universal encoding
        extra='ignore'                 # Ignore other variables in the system env
    )


Trying to load the application configuration

In [3]:
try:
    # no arguments passed
    config = AppConfig()

    print(f"✅ Configuration loaded and validated successfully!\n")
    print(f"Target Environment : {config.automation_env.upper()}")
    print(f"Network Port Connection : {config.central_port}")
    print(f"Execution Dealine Bound : {config.cloud_timeout}s")
    print(f"Target Delivery Destination : {config.slack_webhook_url}")

    # printing Access token securely
    print(f"Cloud Access Token : {config.cloud_access_token}")

    
except Exception as e:
    print(f"❌ Boot Failure: Invalid or missing setting requirements:\n{e}")

✅ Configuration loaded and validated successfully!

Target Environment : PRD
Network Port Connection : 8080
Execution Dealine Bound : 15.5s
Target Delivery Destination : https://slack.com/
Cloud Access Token : **********


Getting the Access Token in text format

In [4]:
try:
    print(f"Access Token in plain Text : {config.cloud_access_token.get_secret_value()}")
except Exception as e:
    print(f"❌ Runtime Error Occured :\n{e}")

Access Token in plain Text : sk-ui12rt456poiujkhqw9876bmnrnjkweqaxcdfertyhnj
